# The alpha gate as a dial: structure -> function

`run_alpha_gate_sweep.py` trains one model per alpha and `alpha_curves.py` melts the
result; **this notebook computes nothing.** It draws the two questions the dial exists
to answer:

1. **does it travel** -- alignment with raw ESM should fall as alpha rises, alignment
   with the train response profile should rise. If those two curves are flat or move
   together, the knob is not a structure/function axis and nothing below it means
   anything.
2. **where does it cross** -- the alpha at which the receptor cloud stops being
   structural and starts being functional.

Run the producer first:

```sh
python scripts/modeling/train/run_alpha_gate_sweep.py --dataset cc hc m2or \
    --mol-source chemberta --nodes onehot --no-legacy \
    --alphas 0 0.05 0.1 0.15 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1.0 \
    --max-parallel 4 --gpus 0 1 2 3
python scripts/analysis/alpha_curves.py --nodes onehot
```

**Why the one-hot series.** With ESM node features alpha=1 does not mean "no
structure": ESM still reaches the receptor cloud through message passing, and it was
measured there at z = 11-18 against ESM where the null is ~2. Only with one-hot nodes
is `1 - alpha` the whole structural path, so only there is the x-axis what its label
says.

In [ ]:
%matplotlib inline
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
RES = ROOT / "results/graph/v8_alpha_gate"

LONG = pd.read_csv(RES / "curves_long.csv")
# An empty variant_tag was written as "" and comes back from CSV as NaN; left alone it
# prints as the string "nan" in every legend entry for cc and hc.
for _c in ("variant_tag", "mol_source", "nodes"):
    if _c in LONG.columns:
        LONG[_c] = LONG[_c].fillna("").astype(str)
LEVELS = (pd.read_csv(RES / "curves_long_levels.csv")
          if (RES / "curves_long_levels.csv").exists() else pd.DataFrame())

GEOMS = ["rsa", "cca", "procrustes"]
GEOM_LABEL = {"rsa": "RSA", "cca": "CCA", "procrustes": "Procrustes"}
REFS = ["esm", "fun"]
REF_LABEL = {"esm": "vs raw ESM  (structure)", "fun": "vs response profile  (function)"}

# Composite encoding, so identity never rests on colour alone: hue = dataset (three
# validated categorical slots), dash = regime. Three hues instead of six keeps every
# pair clear of the colour-vision floors, and the legend plus the numeric table below
# carry the relief the aqua slot's contrast warning asks for.
HUE = {"cc": "#2a78d6", "hc": "#eb6834", "m2or": "#1baf7a"}
DASH = {"transductive": "-", "inductive": "--"}
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#dedddb"

plt.rcParams.update({
    "figure.dpi": 120, "font.size": 9, "axes.titlesize": 9,
    "axes.edgecolor": GRID, "axes.labelcolor": MUTED, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.6, "legend.frameon": False,
})

# One string key per series, written by alpha_curves.py. Selecting on it rather than
# comparing the five identity columns is deliberate: an empty variant_tag round-trips
# through CSV as NaN, NaN != NaN, and a column-wise comparison silently drops every
# series without a variant -- which is all of cc and hc.
KEYS = (LONG[["series", "dataset", "regime", "nodes", "mol_source", "variant_tag"]]
        .drop_duplicates().sort_values(["dataset", "regime", "nodes"])
        .reset_index(drop=True))
# Only the axes that actually VARY across the loaded runs go in the legend label: with
# one node kind on screen "onehot" on every line is noise, but with both on screen a
# label without it silently prints two different models under one name.
# Which secondary axes actually vary across the loaded runs. Only those go in a label:
# with one node kind on screen "onehot" on every line is noise, but with both on screen
# a label without it prints two different models under one name.
# An axis counts as varying only if it varies WITHIN some (dataset, regime) -- that is
# exactly the condition "two lines would collide without it". Measured across the whole
# table instead, variant_tag looks like it varies merely because m2or has one and the
# insects do not, which is already carried by the hue.
VARIES = [c for c in ("nodes", "variant_tag", "mol_source")
          if KEYS.groupby(["dataset", "regime"])[c].nunique().max() > 1]
SECOND = VARIES[0] if VARIES else None
MARKS = ["o", "s", "^", "D"]
MARK = ({v: MARKS[i % len(MARKS)]
         for i, v in enumerate(sorted(set(KEYS[SECOND].astype(str))))} if SECOND else {})
# The encoding has three channels. A second varying secondary axis has nowhere to go,
# so the legend stops describing the encoding and names the series outright -- worse to
# read, but it does not lie by omission.
ENCODED = len(VARIES) <= 1
FIELD = {"nodes": "nodes", "variant_tag": "edges", "mol_source": "mol"}


def name(r):
    extra = " ".join(str(r[c]) for c in VARIES if str(r[c]))
    return f"{r['dataset']} {r['regime'][:5]}" + (f" [{extra}]" if extra else "")


def curve(df, geom, ref, col="value"):
    """Mean over folds (and seeds, if several) at each alpha."""
    q = df[(df.geom == geom) & (df.ref == ref)]
    return q.groupby("alpha")[col].agg(["mean", "sem", "count"]).sort_index()


def style(r):
    return dict(color=HUE.get(r["dataset"], MUTED), linestyle=DASH.get(r["regime"], ":"),
                marker=MARK.get(str(r[SECOND]), "o") if SECOND else "o",
                linewidth=1.8, markersize=4.0, markeredgewidth=0)


def put_legend(fig, extra=(), ncol=None):
    """One legend per FIGURE, placed BELOW it, describing the ENCODING.

    Two things were wrong before: the legend sat inside a data panel, where twelve
    entries covered the very curves they explain, and it listed every combination when
    the encoding is composite -- hue = dataset, dash = regime, marker = the secondary
    axis. Three short groups replace twelve compound entries and double as the key to
    reading the panels."""
    L = plt.Line2D
    if ENCODED:
        h = [L([], [], color=HUE[d], lw=2.0, label=d)
             for d in HUE if d in set(KEYS.dataset)]
        h += [L([], [], color=MUTED, lw=2.0, ls=DASH[r], label=r)
              for r in DASH if r in set(KEYS.regime)]
        h += [L([], [], color=MUTED, lw=0, marker=m, ms=5, label=f"{v} {FIELD[SECOND]}")
              for v, m in MARK.items()]
    else:
        h = [L([], [], label=name(r), **style(r)) for _, r in KEYS.iterrows()]
    h += list(extra)
    fig.legend(handles=h, loc="outside lower center", frameon=False, fontsize=8,
               ncol=ncol or min(len(h), 6), handlelength=2.4, columnspacing=1.8,
               labelcolor=MUTED)


print(f"{len(KEYS)} series, alphas: "
      f"{', '.join(f'{a:g}' for a in sorted(LONG.alpha.unique()))}")
print(f"folds {sorted(LONG.fold.unique().tolist())}   "
      f"seeds {sorted(LONG.seed.unique().tolist())}")
KEYS

## 1. The dial

One panel per (geometry, reference). The left column must **fall** and the right column
must **rise**; that is the whole manipulation check. Raw geometry, not z -- RSA and CCA
are correlations in [0, 1] and read directly, and Procrustes is a residual whose scale
is its own but whose direction is the point. The shaded band is the standard error over
folds.

In [ ]:
fig, axes = plt.subplots(len(GEOMS), len(REFS), figsize=(9.4, 2.6 * len(GEOMS)),
                         squeeze=False, sharex=True, layout="constrained")
for i, g in enumerate(GEOMS):
    for j, ref in enumerate(REFS):
        ax = axes[i][j]
        for _, r in KEYS.iterrows():
            sub = LONG[LONG.series == r["series"]]
            c = curve(sub, g, ref)
            if c.empty:
                continue
            ax.plot(c.index, c["mean"], label=name(r), **style(r))
            if c["sem"].notna().any():
                ax.fill_between(c.index, c["mean"] - c["sem"], c["mean"] + c["sem"],
                                color=HUE.get(r["dataset"], MUTED), alpha=0.12, lw=0)
        ax.set_ylabel(GEOM_LABEL[g] if j == 0 else "")
        if i == 0:
            ax.set_title(REF_LABEL[ref], color=MUTED)
        if i == len(GEOMS) - 1:
            ax.set_xlabel("alpha   (0 = frozen ESM branch only,  1 = graph only)")
fig.suptitle("Receptor-cloud geometry across the gate", ha="left", x=0.01, fontsize=10)
put_legend(fig)
plt.show()

## 2. The same curves against chance

z against each cell's own permutation null: it answers *is this alignment
distinguishable from chance at all*, which a raw correlation cannot. The axis is
symlog because the null's spread shrinks with the receptor count, so M2OR (1237
receptors) reaches z past 1000 where HC (24) tops out near 20.

**Do not read height across panels as effect size.** Within a panel the shape is the
message; across panels only the sign and the crossing are comparable.

In [ ]:
fig, axes = plt.subplots(1, len(GEOMS), figsize=(11.0, 3.1), squeeze=False,
                         layout="constrained")
for j, g in enumerate(GEOMS):
    ax = axes[0][j]
    for _, r in KEYS.iterrows():
        sub = LONG[LONG.series == r["series"]]
        for ref, alpha_ in (("esm", 1.0), ("fun", 0.45)):
            c = curve(sub, g, ref, col="z")
            if c.empty or c["mean"].isna().all():
                continue
            st = style(r) | dict(alpha=alpha_, markersize=3.5)
            ax.plot(c.index, c["mean"], **st)
    ax.set_yscale("symlog", linthresh=1)
    ax.axhline(0, color=MUTED, lw=0.8, zorder=0)
    ax.set_title(GEOM_LABEL[g], color=MUTED)
    ax.set_xlabel("alpha")
    if j == 0:
        ax.set_ylabel("z vs permutation null")
# the reference each curve is scored against is carried by opacity, so it needs an
# entry of its own -- opacity is the one channel a reader cannot name unprompted
put_legend(fig, extra=[plt.Line2D([], [], color=MUTED, lw=2.0, alpha=a, label=l)
                       for a, l in ((1.0, "vs ESM"), (0.45, "vs profile"))])
plt.show()

## 3. Where the cloud changes sides

Each reference curve is min-max scaled inside its own (series, geometry) panel first --
raw RSA against ESM and raw RSA against a response matrix are not on a common scale, so
their literal intersection would be an artefact. After scaling, the plotted quantity is
**structural minus functional**, both in 0..1: above zero the cloud is more ESM-like
than profile-like, below zero the reverse, and the marked point is the crossing.

In [ ]:
fig, axes = plt.subplots(1, len(GEOMS), figsize=(11.0, 3.1), squeeze=False,
                         sharey=True, layout="constrained")
rows = []
for j, g in enumerate(GEOMS):
    ax = axes[0][j]
    for _, r in KEYS.iterrows():
        sub = LONG[LONG.series == r["series"]]
        m = pd.DataFrame({ref: curve(sub, g, ref)["mean"] for ref in REFS}).dropna()
        if len(m) < 3:
            continue
        n = (m - m.min()) / (m.max() - m.min()).replace(0, np.nan)
        d = (n["esm"] - n["fun"]).dropna()
        ax.plot(d.index, d.values, **style(r), label=name(r))
        if len(d) > 1 and d.iloc[0] * d.iloc[-1] < 0:
            i = int(np.argmax(np.sign(d.values) != np.sign(d.iloc[0])))
            x0, x1, y0, y1 = d.index[i - 1], d.index[i], d.iloc[i - 1], d.iloc[i]
            xc = x0 - y0 * (x1 - x0) / (y1 - y0)
            ax.plot([xc], [0], marker="D", ms=5, color=HUE.get(r["dataset"], MUTED),
                    markeredgecolor="white", markeredgewidth=0.8, zorder=5)
            rows.append({"series": name(r), "geom": g, "alpha_cross": round(float(xc), 3)})
    ax.axhline(0, color=MUTED, lw=0.9, zorder=0)
    ax.set_title(GEOM_LABEL[g], color=MUTED)
    ax.set_xlabel("alpha")
    if j == 0:
        ax.set_ylabel("structural - functional\n(scaled within panel)")
put_legend(fig)
plt.show()

pd.DataFrame(rows).pivot_table(index="series", columns="geom", values="alpha_cross") \
    if rows else print("no curve crosses -- the cloud never changes sides")

## 4. What the dial costs

The same alphas, scored by the pipeline's own head on held-out rows. The horizontal
lines are the two references that are not the gate: `boost` is XGBoost on
[raw ESM || molecule], `legacy` is the pre-v8 graph.

Read this as a **shape, not a scoreboard.** These rows carry one seed, and the head
lottery alone moves the metric by ~0.02 -- the numbers with error bars are in
`headline_table.py`.

In [ ]:
# The metric of record follows the TASK, and the datasets do not share one: cc/hc are
# regression (R2) and m2or is classification (AUROC). One metric for the whole figure
# leaves an empty panel, so each panel picks its own and says so on its own axis. These
# are separate measures on separate panels, never two scales on one.
def metric_of(ds):
    sub = LONG[LONG.dataset == ds]
    return next((m for m in ("R2", "AUROC")
                 if m in sub.columns and sub[m].notna().any()), None)


ds_list = sorted(set(KEYS.dataset))
fig, axes = plt.subplots(1, len(ds_list), figsize=(3.7 * len(ds_list), 3.1),
                         squeeze=False, layout="constrained")
for j, ds in enumerate(ds_list):
    ax, MET = axes[0][j], metric_of(ds)
    if MET is None:
        ax.set_axis_off(); ax.set_title(f"{ds}: no prediction column", color=MUTED)
        continue
    for _, r in KEYS[KEYS.dataset == ds].iterrows():
        sub = LONG[LONG.series == r["series"]]
        c = sub[sub.geom == GEOMS[0]].groupby("alpha")[MET].mean().sort_index()
        if c.empty:
            continue
        ax.plot(c.index, c.values, **style(r), label=name(r))
        if len(LEVELS) and "series" in LEVELS.columns:
            for lvl, ls in (("boost", ":"), ("legacy", "-.")):
                v = LEVELS[(LEVELS.level == lvl) & (LEVELS.series == r["series"])]
                if len(v) and MET in v.columns and pd.notna(v.iloc[0][MET]):
                    ax.axhline(v.iloc[0][MET], ls=ls, lw=1.0,
                               color=HUE.get(ds, MUTED), alpha=0.55)
    ax.set_title(ds, color=MUTED)
    ax.set_xlabel("alpha")
    ax.set_ylabel(MET)
put_legend(fig, extra=[plt.Line2D([], [], color=MUTED, ls=ls, lw=1.2, label=l)
                       for l, ls in (("boost [ESM||mol]", ":"), ("pre-v8 graph", "-."))])
plt.show()

## 5. The numbers

The endpoints, the travel and the monotonicity of every curve -- the table view that
backs the plots above, and the thing to read first if a panel looks ambiguous.
`mono` is the Spearman of the curve against alpha: it should be near **-1** for `esm`
and near **+1** for `fun`.

In [ ]:
from scipy.stats import spearmanr

rows = []
for _, r in KEYS.iterrows():
    sub = LONG[LONG.series == r["series"]]
    for g in GEOMS:
        for ref in REFS:
            c = curve(sub, g, ref)["mean"]
            if len(c) < 3:
                continue
            rows.append({"series": name(r), "geom": GEOM_LABEL[g], "ref": ref,
                         "a=0": c.iloc[0], "a=1": c.iloc[-1],
                         "travel": c.max() - c.min(),
                         "mono": spearmanr(c.index.to_numpy(), c.to_numpy()).statistic})
tab = pd.DataFrame(rows)
bad = tab[((tab.ref == "esm") & (tab.mono > -0.7)) | ((tab.ref == "fun") & (tab.mono < 0.7))]
print(f"{len(bad)} of {len(tab)} curves fail the direction check"
      + (":" if len(bad) else " -- the dial behaves everywhere"))
display(bad if len(bad) else tab.head(0))
tab.pivot_table(index=["series", "geom"], columns="ref",
                values=["a=0", "a=1", "travel", "mono"]).round(3)

## Reading this

**Section 1 gates everything else.** If the left column does not fall and the right does
not rise, the gate is not a structure/function axis on that dataset and section 4 is
uninterpretable -- a prediction curve over a knob that moves nothing is a plot of noise.

**Then section 3.** A crossing near alpha = 0 means the graph takes over almost
immediately: the frozen structural branch has to be weighted very heavily before it
shows in the geometry. That was the finding the RMS normalisation fixed -- when the two
branches were scaled by total energy, ESM's ~94% common mean meant the crossover sat at
alpha ~ 0.05 and every alpha above 0.25 was geometrically the same model.

**Section 4 last, and gently.** One seed, and the head lottery is ~0.02 wide. It shows
whether the cost of turning the dial is smooth or cliff-like; it does not rank models.
The ranked table with intervals over folds x seeds is `headline_table.py`.